In [3]:
# --------------------------------------------
# 0. Libraries-ka loo baahan yahay
# --------------------------------------------

# numpy iyo pandas waxaa loo isticmaalaa data handling
import numpy as np
import pandas as pd

# data splitting
from sklearn.model_selection import train_test_split

# text -> numeric conversion
from sklearn.feature_extraction.text import TfidfVectorizer

# machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB

# model evaluation metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# random state si natiijo isku mid ah mar kasta loo helo
RANDOM_STATE = 42

In [4]:
# --------------------------------------------
# 1. Dataset Loading
# --------------------------------------------

# dataset-ka SMS spam
df = pd.read_csv("mail_l7_dataset.csv")

# NaN values waxaan ku badalaynaa empty string
df = df.where(pd.notnull(df), "")

# label cleaning
df["Category"] = df["Category"].str.lower().str.strip()

# label encoding
# spam = 0
# ham = 1
df["Category"] = df["Category"].map({"spam":0, "ham":1})

# 5 rows ee ugu horeeya
df.head()

,Category,Message
0,1,"Go until jurong point, crazy.. Available only ..."
1,1,Ok lar... Joking wif u oni...
2,0,Free entry in 2 a wkly comp to win FA Cup fina...
3,1,U dun say so early hor... U c already then say...
4,1,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
# --------------------------------------------
# 2. Feature iyo Target
# --------------------------------------------

# X = fariimaha
X = df["Message"].astype(str)

# y = label (spam ama ham)
y = df["Category"].astype(int)

In [6]:
# --------------------------------------------
# 3. Train / Test Split
# --------------------------------------------

# 80% training
# 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("Training size:", X_train.shape[0])
print("Testing size:", X_test.shape[0])

Training size: 4457
Testing size: 1115


In [7]:
# --------------------------------------------
# 4. Text -> Numeric (TF-IDF)
# --------------------------------------------

tfidf = TfidfVectorizer(
    min_df=1,
    stop_words="english",
    lowercase=True
)

# training data vectorization
X_train_vec = tfidf.fit_transform(X_train)

# test data vectorization
X_test_vec = tfidf.transform(X_test)

print("Train shape:", X_train_vec.shape)
print("Test shape:", X_test_vec.shape)

Train shape: (4457, 7440)
Test shape: (1115, 7440)


In [8]:
# --------------------------------------------
# 5. Model Training
# --------------------------------------------

# Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_model.fit(X_train_vec, y_train)

lr_predictions = lr_model.predict(X_test_vec)


# Random Forest
rf_model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf_model.fit(X_train_vec, y_train)

rf_predictions = rf_model.predict(X_test_vec.toarray())


# Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

nb_predictions = nb_model.predict(X_test_vec)

In [9]:
# --------------------------------------------
# 6. Evaluation Functions
# --------------------------------------------

def show_metrics(name, y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=0)
    rec = recall_score(y_true, y_pred, pos_label=0)
    f1 = f1_score(y_true, y_pred, pos_label=0)

    print("\n", name)
    print("Accuracy :", round(acc,3))
    print("Precision:", round(prec,3))
    print("Recall   :", round(rec,3))
    print("F1 Score :", round(f1,3))


def show_confusion(name, y_true, y_pred):

    cm = confusion_matrix(y_true, y_pred, labels=[1,0])

    df_cm = pd.DataFrame(
        cm,
        index=["Actual Ham", "Actual Spam"],
        columns=["Pred Ham", "Pred Spam"]
    )

    print("\nConfusion Matrix -", name)
    print(df_cm)

In [10]:
# --------------------------------------------
# 7. Model Evaluation
# --------------------------------------------

show_metrics("Logistic Regression", y_test, lr_predictions)
show_confusion("Logistic Regression", y_test, lr_predictions)

show_metrics("Random Forest", y_test, rf_predictions)
show_confusion("Random Forest", y_test, rf_predictions)

show_metrics("Naive Bayes", y_test, nb_predictions)
show_confusion("Naive Bayes", y_test, nb_predictions)


 Logistic Regression
Accuracy : 0.968
Precision: 1.0
Recall   : 0.758
F1 Score : 0.863

Confusion Matrix - Logistic Regression
             Pred Ham  Pred Spam
Actual Ham        966          0
Actual Spam        36        113

 Random Forest
Accuracy : 0.983
Precision: 1.0
Recall   : 0.872
F1 Score : 0.932

Confusion Matrix - Random Forest
             Pred Ham  Pred Spam
Actual Ham        966          0
Actual Spam        19        130

 Naive Bayes
Accuracy : 0.977
Precision: 1.0
Recall   : 0.826
F1 Score : 0.904

Confusion Matrix - Naive Bayes
             Pred Ham  Pred Spam
Actual Ham        966          0
Actual Spam        26        123


In [11]:
# --------------------------------------------
# 8. Sanity Check
# --------------------------------------------

def label_to_text(v):
    return "Spam" if v == 0 else "Ham"


messages = [
    "Free entry in 2 a weekly competition!",
    "I will meet you at the cafe tomorrow",
    "Congratulations, you won a free ticket"
]

for msg in messages:

    msg_vec = tfidf.transform([msg])

    lr_p = int(lr_model.predict(msg_vec)[0])
    rf_p = int(rf_model.predict(msg_vec.toarray())[0])
    nb_p = int(nb_model.predict(msg_vec)[0])

    print("\nMessage:", msg)
    print("LR:", label_to_text(lr_p))
    print("RF:", label_to_text(rf_p))
    print("NB:", label_to_text(nb_p))


Message: Free entry in 2 a weekly competition!
LR: Ham
RF: Ham
NB: Spam

Message: I will meet you at the cafe tomorrow
LR: Ham
RF: Ham
NB: Ham

Message: Congratulations, you won a free ticket
LR: Ham
RF: Ham
NB: Ham
